In [1]:
import pandas as pd
import ast
import argparse
import os
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import math


## load metadata

In [14]:
PRECLINICAL_METADATA_PATH = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/03_IE_ner/data/animal_studies_with_drug_disease/animal_studies_metadata_595768.csv"
PRECLINICAL_ANNOTATIONS_PATH = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_preclin.csv"

# load data
metadata_df = pd.read_csv(PRECLINICAL_METADATA_PATH)
annotations_df = pd.read_csv(PRECLINICAL_ANNOTATIONS_PATH)[['PMID', 
       'merged_mondo_label', 'merged_mondo_termid',
       'merged_umls_label', 'merged_umls_termid']]

print(f"Initial shapes -> metadata: {metadata_df.shape}, annotations: {annotations_df.shape}")

# deduplicate both by PMID
annotations_df = annotations_df.drop_duplicates(subset=["PMID"])
print(f"After dedup annotations: {annotations_df.shape}")

metadata_df = metadata_df.drop_duplicates(subset=["PMID"])
print(f"After dedup metadata: {metadata_df.shape}")

# merge on PMID (left join to keep all annotation records)
merged_matadata_df = pd.merge(annotations_df, metadata_df, on="PMID", how="left")
print(f"After merge: {merged_matadata_df.shape}")

# check unique PMIDs
print(f"Unique PMIDs after merge: {merged_matadata_df['PMID'].nunique()}")


Initial shapes -> metadata: (595768, 7), annotations: (540999, 5)
After dedup annotations: (540999, 5)
After dedup metadata: (581385, 7)
After merge: (540999, 11)
Unique PMIDs after merge: 540999


In [15]:
merged_matadata_df[merged_matadata_df['PMID']==2434820]

,PMID,merged_mondo_label,merged_mondo_termid,merged_umls_label,merged_umls_termid,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,year,journal_name,publication_type,title


In [16]:
merged_matadata_df[merged_matadata_df["unique_interventions_linkbert_predictions"].astype(str).str.contains("fampridine", case=False, na=False)]


,PMID,merged_mondo_label,merged_mondo_termid,merged_umls_label,merged_umls_termid,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,year,journal_name,publication_type,title
154179,22560931,multiple sclerosis,MONDO:0005301,Dalfampridine|non-specific kv channel blocker,C0000477|-1,multiple sclerosis,dalfampridine|non-specific kv channel blocker|...,2012.0,Neurobiology of disease,"Journal Article+Research Support, N.I.H., Extr...",K+ channel alterations in the progression of e...
182355,37567585,multiple sclerosis,MONDO:0005301,Dalfampridine,C0000477,multiple sclerosis,fampridine|dalfampridine,2024.0,Journal of chromatographic science,Journal Article,Ultra-High Performance Liquid Chromatography M...
241810,37242713,Lambert-Eaton myasthenic syndrome,MONDO:0018556,Acetaminophen|AMIFAMPRIDINE|N-acetyltransferase-1,C0000970|C0046948|C0259190,lambert-eaton myasthenic syndrome,acetaminophen|amifampridine|n-acetyltransferase 2,2023.0,Pharmaceutics,Journal Article,Investigation of N-Acetyltransferase 2-Mediate...
242878,21501202,Lambert-Eaton myasthenic syndrome,MONDO:0018556,AMIFAMPRIDINE,C0046948,lambert-eaton myasthenic syndrome,", 4-dap|3 , 4-dap|amifampridine|3 , 4-diaminop...",2012.0,Journal of clinical pharmacy and therapeutics,"Journal Article+Research Support, Non-U.S. Gov't",Content variability of active drug substance i...
309119,33738893,multiple sclerosis,MONDO:0005301,[15]n] dalfampridine|[[15] n] dalfampridine,-1|-1,multiple sclerosis,[15]n] dalfampridine|[[15] n] dalfampridine,2021.0,Chemphyschem : a European journal of chemical ...,"Journal Article+Research Support, Non-U.S. Gov't",Synthesis and [15] N NMR Signal Amplification ...
381372,38934397,spinal cord injury|chronic spinal cord injury,MONDO:0043797|-1,Diosgenin|Chondroitin ABC Lyase|Granulocyte co...,C0012497|C0008455|C0079459|-1|-1|C0059438|-1|C...,| spinal cord injury|chronic spinal cord injur...,diosgenin|chondroitinase abc|granulocyte-colon...,2025.0,Neural regeneration research,Journal Article,Pharmacological intervention for chronic phase...


## load annotations

In [17]:
def build_annotation_files(base_dir):
    return {
        'animal_sex': (
            os.path.join(base_dir, "regex/sex_doc_level_predictions.csv"),
            {'prediction_encoded_label': 'animal_sex'},
            False
        ),
        'animal_species': (
            os.path.join(base_dir, "regex/species_doc_level_predictions.csv"),
            {'prediction_encoded_label': 'animal_species'},
            False
        ),
        'animal_age': (
            os.path.join(base_dir, "age/age_unsloth_meta_llama_3.1_8b_doc_level_predictions_mapped.csv"),
            {
                'age_classification': 'animal_age_class',
                'prediction_normalized_age': 'animal_age'
            },
            False
        ),
        'rigor_blinding': (
            os.path.join(base_dir, "regex/blinding_doc_level_predictions.csv"),
            {'prediction_encoded_label': 'rigor_blinding'},
            False
        ),
        'rigor_randomization': (
            os.path.join(base_dir, "regex/randomization_doc_level_predictions.csv"),
            {'prediction_encoded_label': 'rigor_randomization'},
            False
        ),
        'rigor_welfare': (
            os.path.join(base_dir, "regex/welfare_doc_level_predictions.csv"),
            {'prediction_encoded_label': 'rigor_welfare'},
            False
        ),
        'assay_type': (
            os.path.join(base_dir, "regex/assay_doc_level_predictions.csv"),
            {'prediction_encoded_label': 'assay_type'},
            False
        ),
        'animal_strain': (
            os.path.join(base_dir, "strain/strain_predictions_normalized.csv"),
            {'animal_strain_norm_family': 'animal_strain'},
            False
        ),
        'animal_number': (
            os.path.join(base_dir, "animals_nr/animals_nr_predictions_numeric.csv"),
            {'prediction_encoded_label': 'animal_number'},
            False
        ),
        'sample_size': (
            os.path.join(base_dir, "regex/sample_size_doc_level_predictions.csv"),
            {'prediction_encoded_label': 'sample_size'},
            False
        )
        ,
        'first_author_affiliation': (
            os.path.join(base_dir, "author_affiliations_pubmed_mapped.csv"),
            {'first_author_country': 'first_author_country'},
            False
        )
    }

def load_annotation(file_path, col_rename_map, parse_list=False):
    """
    Load a single annotation CSV file and select/rename columns as needed.

    Args:
        file_path (str): Path to the CSV file containing predictions.
        col_rename_map (dict): Mapping from column names in the file to desired output names.
        parse_list (bool): Whether to parse stringified lists into actual strings (e.g. ['a', 'b'] → 'a, b').

    Returns:
        pd.DataFrame: DataFrame with 'PMID' and renamed annotation columns.
    """
    print(f'Loading annotation from: {file_path}')
    df = pd.read_csv(file_path, low_memory=False)

    # Select only required columns
    required_cols = ['PMID'] + list(col_rename_map.keys())
    if missing := (set(required_cols) - set(df.columns)):
        raise ValueError(f"Missing columns in {file_path}: {missing}")
    
    df = df[required_cols].rename(columns=col_rename_map)

    # Optionally convert stringified lists to comma-separated strings
    if parse_list:
        for col in col_rename_map.values():
            df[col] = df[col].apply(
                lambda x: ', '.join(ast.literal_eval(x)) if pd.notna(x) else x
            )

    return df

In [19]:
base_annotation_dir = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions"
ANNOTATION_FILES = build_annotation_files(base_annotation_dir)

merged_annotations_df = None  # start empty

for label_group, (file_path, col_rename_map, parse_list) in ANNOTATION_FILES.items():
    print(f"\nProcessing annotation group: {label_group}")
    if "age" in label_group:
        print("Skipping age")
        continue
    # load one annotation file
    annot_df = load_annotation(file_path, col_rename_map, parse_list=parse_list)
    print(f"Annotations size: {annot_df.shape}")
    
    # drop duplicates (if any)
    annot_df = annot_df.drop_duplicates(subset=["PMID"])
    annot_df["PMID"] = annot_df["PMID"].astype(str)
    print(f"After dedup: {annot_df.shape}")
    
    # merge into cumulative dataframe
    if merged_annotations_df is None:
        merged_annotations_df = annot_df
    else:
        merged_annotations_df = pd.merge(merged_annotations_df, annot_df, on="PMID", how="left")
    
    print(f"Current merged size: {merged_annotations_df.shape}")

# fill missing values after all merges
merged_annotations_df = merged_annotations_df.fillna("not reported")


Processing annotation group: animal_sex
Loading annotation from: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/regex/sex_doc_level_predictions.csv
Annotations size: (371832, 2)
After dedup: (371832, 2)
Current merged size: (371832, 2)

Processing annotation group: animal_species
Loading annotation from: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/regex/species_doc_level_predictions.csv
Annotations size: (371832, 2)
After dedup: (371832, 2)
Current merged size: (371832, 3)

Processing annotation group: animal_age
Skipping age

Processing annotation group: rigor_blinding
Loading annotation from: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/regex/blinding_doc_level_predictions.csv
Annotations size: (371832, 2)
After dedup: (371832, 2)
Current merged size: (371832, 4)

Processing annotation group: rigor_randomization
Loading annotation from: /shares/animalwelfare.crs.uzh

In [21]:
country_df = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/author_affiliations_pubmed_mapped.csv")
country_df

,PMID,first_author_affiliation,authors,all_affiliations,first_author_geolocation,first_author_country
0,10024109,Ludwig Boltzmann Institute for Experimental an...,"Schlag G, Redl H, Davies J, Scannon P",Ludwig Boltzmann Institute for Experimental an...,"Wien, Austria",Austria
1,10028928,"Department of Cardiology, Cardiovascular Resea...","Verduyn S C, Vos M A, Leunissen H D, van Opsta...","Department of Cardiology, Cardiovascular Resea...","Maastricht, Limburg, Netherlands",Netherlands
2,10028929,"Department of Cardiology, Cardiovascular Resea...","Takase H, Oemar B S, Pech M, Lüscher T F","Department of Cardiology, Cardiovascular Resea...","New York, NY, USA",USA
3,10028938,"Systems Biology Unit, Glaxo-Wellcome Research ...","Louttit J B, Hunt A A, Maxwell M P, Drew G M","Systems Biology Unit, Glaxo-Wellcome Research ...","Stevenage, Hertfordshire, UK",UK
4,10028944,"Department of Medical Cardiology, Royal Infirm...","Wolk R, Cobbe S M, Kane K A, Hicks M N","Department of Medical Cardiology, Royal Infirm...","Glasgow, UK",UK
...,...,...,...,...,...,...
371855,20687175,"Laboratory of Epidemiology, Demography and Bio...","Shahar Avner, Patel Kushang V, Semba Richard D...","Laboratory of Epidemiology, Demography and Bio...","Bethesda, MD, USA",USA
371856,21801587,Department of Anesthesiology and Critical Care...,"Armstead William M, Raghupathi Ramesh",Department of Anesthesiology and Critical Care...,"Philadelphia, PA, USA",USA
371857,25188549,"Anaesthetics, Pain Medicine and Intensive Care...","Campos-Pires Rita, Armstrong Scott P, Sebastia...","Anaesthetics, Pain Medicine and Intensive Care...","London, UK",UK
371858,30673266,NaN,"Dong Xinyue, Gao Jin, Zhang Can Yang, Hayworth...",NaN,unlabeled,unlabeled


In [22]:
merged_annotations_df.head()

,PMID,animal_sex,animal_species,rigor_blinding,rigor_randomization,rigor_welfare,assay_type,animal_strain,animal_number,sample_size,first_author_country
0,1000129,sex-not-reported,mouse,blinding-not-reported,randomization-not-reported,welfare-not-reported,not reported,not reported,not reported,sample-size-not-reported,unlabeled
1,1000338,sex-not-reported,rat,blinding-not-reported,randomization-present,welfare-not-reported,"Histology, Molecular & Cellular",not reported,not reported,sample-size-not-reported,unlabeled
2,10021294,sex-not-reported,mouse,blinding-not-reported,randomization-not-reported,welfare-not-reported,not reported,not reported,45.0,sample-size-not-reported,USA
3,10021348,sex-not-reported,mouse,blinding-not-reported,randomization-not-reported,welfare-not-reported,"Histology, Molecular & Cellular",not reported,not reported,sample-size-not-reported,UK
4,10022166,sex-female,rat,blinding-not-reported,randomization-not-reported,welfare-not-reported,"Histology, Molecular & Cellular",Sprague-Dawley,not reported,sample-size-not-reported,USA


In [23]:
merged_annotations_df.shape

(371832, 11)

## merge metadata and annotations

In [25]:
merged_annotations_df["PMID"] = merged_annotations_df["PMID"].astype(str)
merged_matadata_df["PMID"] = merged_matadata_df["PMID"].astype(str)

# Merge on PMID
dataset_fulltext = pd.merge(merged_annotations_df, merged_matadata_df, on="PMID", how="left")
dataset_fulltext.shape

(371832, 21)

In [26]:
dataset_fulltext.head()

,PMID,animal_sex,animal_species,rigor_blinding,rigor_randomization,rigor_welfare,assay_type,animal_strain,animal_number,sample_size,...,merged_mondo_label,merged_mondo_termid,merged_umls_label,merged_umls_termid,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,year,journal_name,publication_type,title
0,1000129,sex-not-reported,mouse,blinding-not-reported,randomization-not-reported,welfare-not-reported,not reported,not reported,not reported,sample-size-not-reported,...,audiogenic si|audiogenic seizures,-1|MONDO:0015644,Naloxone|Morphine,C0027358|C0026549,audiogenic si|audiogenic seizures,naloxone|morphine,1976.0,British journal of pharmacology,Journal Article,Effect of morphine and naloxone on priming-ind...
1,1000338,sex-not-reported,rat,blinding-not-reported,randomization-present,welfare-not-reported,"Histology, Molecular & Cellular",not reported,not reported,sample-size-not-reported,...,vitamin D deficiency,MONDO:0100471,VITAMIN D,C3714503,vitamin d deficient,vitamin d,1976.0,Calcified tissue research,"Journal Article+Research Support, U.S. Gov't, ...",A morphometric investigation of the duodenal m...
2,10021294,sex-not-reported,mouse,blinding-not-reported,randomization-not-reported,welfare-not-reported,not reported,not reported,45.0,sample-size-not-reported,...,ovarian carcinoma|ovarian neoplasm,MONDO:0005140|MONDO:0021068,"Il12a protein, mouse|Interleukin 12|Edodekin a...",C1700202|C0123759|C3665495|C1121403,epithelial ovarian cancer|ovarian cancer|ovari...,murine il-12|il-12|murine interleuken (il)-12,1999.0,Gynecologic oncology,"Journal Article+Research Support, Non-U.S. Gov...",Effects of IL-12 on human ovarian tumors engra...
3,10021348,sex-not-reported,mouse,blinding-not-reported,randomization-not-reported,welfare-not-reported,"Histology, Molecular & Cellular",not reported,not reported,sample-size-not-reported,...,obsolete cartilage disease,MONDO:0005569,Growth Differentiation Factor 5,C0253373,chondrody,gdf 5|gdf-5,1999.0,"Development (Cambridge, England)","Journal Article+Research Support, Non-U.S. Gov't",Mechanisms of GDF-5 action during skeletal dev...
4,10022166,sex-female,rat,blinding-not-reported,randomization-not-reported,welfare-not-reported,"Histology, Molecular & Cellular",Sprague-Dawley,not reported,sample-size-not-reported,...,congenital diaphragmatic hernia,MONDO:0005711,Vitamin E,C0042874,congenital diaphragmatic hernia,vitamin e,1999.0,Journal of pediatric surgery,"Journal Article+Research Support, Non-U.S. Gov...",Prenatal vitamin E treatment improves lung gro...


In [27]:
dataset_fulltext.columns

Index(['PMID', 'animal_sex', 'animal_species', 'rigor_blinding',
       'rigor_randomization', 'rigor_welfare', 'assay_type', 'animal_strain',
       'animal_number', 'sample_size', 'first_author_country',
       'merged_mondo_label', 'merged_mondo_termid', 'merged_umls_label',
       'merged_umls_termid', 'unique_conditions_linkbert_predictions',
       'unique_interventions_linkbert_predictions', 'year', 'journal_name',
       'publication_type', 'title'],
      dtype='object')

In [29]:
dataset_fulltext[dataset_fulltext['PMID']=="26114502"]["merged_mondo_label"]

154766    neuronitis
Name: merged_mondo_label, dtype: object

In [30]:
base_annotation_dir

'/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions'

In [32]:
dataset_fulltext.to_csv(f"{base_annotation_dir}/full_text_combined_all_annotations_metadata.csv",index=False)